---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Evaluating Models


🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 1. Starting out

We start off as we usually do. Let's import some things that will be useful.

The packages below are already included in the course environment. If needed, run `uv sync` in your terminal in the `wa` folder, then select the course's `.venv` Python environment as your notebook kernel.

Run the cells in order; later cells use things we created earlier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

%matplotlib inline

# notebooks can start in wa, scripts, or homework; let's find the course folder
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')

data_folder = course_folder / 'data'
print('Course folder:', course_folder)

# 2. Data

We're going to use a mail response dataset from a real direct marketing campaign. A copy is already in `data/mailing.csv`, so there's nothing to download.

Each row represents someone who received a request to make a charitable donation. Our goal is to predict whether they gave during this campaign: `class=1` means YES, and `class=0` means NO.

Here are the columns in OUR copy of the data:

| Column | Description |
| --- | --- |
| `Income` | Household income code, not a dollar amount |
| `Firstdate` | Code for the date of the first gift |
| `Lastdate` | Code for the date of the most recent gift |
| `Amount` | Average donation amount over all periods, including zeros |
| `rfaf2` | Donation frequency code |
| `glast` | Amount of the last gift |
| `gavr` | Average gift amount |
| `rfaa2_G`, `rfaa2_E`, `rfaa2_F` | Donation-amount categories, already turned into 0/1 columns |
| `pepstrfl_0` | An already-encoded 0/1 column from the star-donor flag |
| `class` | Our target: 1 if they gave in this campaign, 0 otherwise |

The category columns have already been encoded for us: a 1 means that the row belongs to that column's category. Don't treat codes like `Income` as literal dollar amounts!

Let's read our data in and put the target variable in `Y` and all the other features in `X`.

In [ ]:
# read the local CSV
data = pd.read_csv(data_folder / 'mailing.csv')

# X contains our predictors; Y contains the answer we want to predict
X = data.drop(columns=['class'])
Y = data['class']

In [ ]:
len(data)

In [ ]:
# how many rows and predictors do we have?
X.shape

In [ ]:
data.head()

In [ ]:
X.head()

In [ ]:
Y.head()

# 3. Overfitting

In the previous notebook, we checked a tree on the SAME cars we used to train it. That tells us how well it learned those cars, not how well it predicts new ones.

Let's look at this more carefully with our donation data.

## 3.1 Keeping some answers out of sight

We'll create THREE groups of rows:

- **Training (60%)**: The model learns from these rows.
- **Validation (20%)**: We check our choices here, like how deep to make the tree.
- **Final test (20%)**: We leave these rows alone until we've finished choosing the model.

Why not keep checking the test data while making changes? Because then our choices start depending on those answers too! It stops being a fresh check.

`stratify=Y` keeps roughly the same proportion of donors in each group. `random_state=42` makes the split repeatable; 42 itself isn't special.

In [ ]:
# first, set aside 20% for the FINAL test
X_development, X_test, Y_development, Y_test = train_test_split(
    X, Y, test_size=0.20, stratify=Y, random_state=42
)

# 25% of the remaining 80% is 20% of the original data
X_train, X_validation, Y_train, Y_validation = train_test_split(
    X_development, Y_development,
    test_size=0.25, stratify=Y_development, random_state=42
)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_validation))
print('Final test rows:', len(X_test))

## 3.2 Looking good on the training data

Let's create a decision tree with no depth limit, fit it to the training rows, and predict those SAME rows.

In [ ]:
# create an empty, unlearned tree
tree = DecisionTreeClassifier(criterion='entropy', random_state=42)

# fit() learns from the training rows ONLY
tree.fit(X_train, Y_train)
Y_train_predicted = tree.predict(X_train)

# keep the answers and predictions next to each other, without changing X
training_predictions = X_train.assign(actual=Y_train, predicted=Y_train_predicted)
training_predictions.head(10)

In [ ]:
# actual answers first, then our predictions
training_accuracy = accuracy_score(Y_train, Y_train_predicted)
print(f'Training accuracy: {training_accuracy:.1%}')

That's a pretty high accuracy. Is it? Let's check the base rate.

In [ ]:
# Y is 1 for donors, so its mean is the fraction who donated
Y_train.mean()

In [ ]:
# what if we ALWAYS predicted "no donation"?
training_baseline_accuracy = (Y_train == 0).mean()
print(f'Always predict no donation (training): {training_baseline_accuracy:.1%}')

About 95% of these people did NOT donate. A model that always says "no donation" gets about 95% accuracy, without learning anything useful about donors!

Our tree looks even better on the training rows. But it may have learned very specific rules that don't work well on new rows. This is **overfitting**.

## 3.3 Looking at rows the tree did NOT learn from

Let's use the SAME fitted tree on our validation data. No new `fit()` call here!

In [ ]:
Y_validation_predicted = tree.predict(X_validation)
validation_accuracy = accuracy_score(Y_validation, Y_validation_predicted)
validation_baseline_accuracy = (Y_validation == 0).mean()

print(f'Tree training accuracy: {training_accuracy:.1%}')
print(f'Tree validation accuracy: {validation_accuracy:.1%}')
print(f'Always predict no donation (validation): {validation_baseline_accuracy:.1%}')

That's a pretty big drop! On new rows, our tree does WORSE than always saying "no donation."

Accuracy counts both kinds of correct answer. But if we're trying to find donors, we also want to know WHO we're getting right or wrong.

A **confusion matrix** just puts these counts into a small table. Rows are the actual answers; columns are our predictions. Look at the two cells OFF the diagonal: those are our mistakes.

In [ ]:
validation_confusion = pd.DataFrame(
    confusion_matrix(Y_validation, Y_validation_predicted, labels=[0, 1]),
    index=['Actually no donation', 'Actually donation'],
    columns=['Predict no donation', 'Predict donation']
)
validation_confusion

Two useful questions:

- **Precision**: Of the people we predicted would donate, what fraction actually donated?
- **Recall**: Of ALL the people who actually donated, what fraction did we find?

A false positive means we predicted a donation that didn't happen. A false negative means we missed someone who did donate. In a mailing campaign, those mistakes can have different costs.

In [ ]:
validation_precision = precision_score(Y_validation, Y_validation_predicted, zero_division=0)
validation_recall = recall_score(Y_validation, Y_validation_predicted, zero_division=0)

print(f'Precision (donors): {validation_precision:.1%}')
print(f'Recall (donors): {validation_recall:.1%}')

## 3.4 Cross-validation

Our validation score comes from one particular split. Let's also use **5-fold cross-validation** on the TRAINING rows.

We divide those rows into five groups. Each time, a fresh tree learns from four groups and predicts the remaining one. We do this five times, so every group gets a turn being left out.

`StratifiedKFold` keeps roughly the same donor proportion in each fold. Neither our validation rows nor our final test rows take part in this step.

In [ ]:
cv_tree = DecisionTreeClassifier(criterion='entropy', random_state=42)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# scikit-learn fits a fresh copy of the tree for each fold
cross_fold_accuracies = cross_val_score(
    cv_tree, X_train, Y_train, scoring='accuracy', cv=cv_splitter
)
average_cross_fold_accuracy = cross_fold_accuracies.mean()

In [ ]:
for fold_number, fold_accuracy in enumerate(cross_fold_accuracies, start=1):
    print(f'Fold {fold_number}: {fold_accuracy:.1%}')

In [ ]:
print(f'Mean cross-validation accuracy: {average_cross_fold_accuracy:.1%}')

Which accuracy do you "trust" more: the training accuracy, or an accuracy measured on rows the fitted tree did NOT see? Why?

Cross-validation doesn't make the model better by itself. It helps us check it. And a high-looking accuracy still needs a sensible baseline!

# 4. Creating a simple learning curve

What happens if we give our tree more training data?

We'll change the SIZE of the training sample, keeping the tree settings and validation rows the SAME. For each sample, we'll record accuracy on that sample AND on the fixed validation set.

What list should we use for the x-axis? What about the y-axis?

In [ ]:
training_percentages = np.linspace(0.1, 1.0, 10)
training_sizes = []
learning_train_accuracies = []
learning_validation_accuracies = []

for training_percentage in training_percentages:
    if training_percentage == 1.0:
        # use ALL our training rows for the last point
        X_small, Y_small = X_train, Y_train
    else:
        X_small, _, Y_small, _ = train_test_split(
            X_train, Y_train, train_size=training_percentage,
            stratify=Y_train, random_state=42
        )

    learning_tree = DecisionTreeClassifier(criterion='entropy', random_state=42)
    learning_tree.fit(X_small, Y_small)

    training_sizes.append(len(X_small))
    learning_train_accuracies.append(learning_tree.score(X_small, Y_small))
    learning_validation_accuracies.append(learning_tree.score(X_validation, Y_validation))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(training_sizes, learning_train_accuracies, 'o-', color='#4C72B0', label='Training')
ax.plot(training_sizes, learning_validation_accuracies, 's--', color='#DD8452',
        markerfacecolor='white', label='Validation')
ax.axhline(validation_baseline_accuracy, color='#666666', linestyle=':',
           label='Always no donation (validation)')
ax.set_title('Learning curve: changing the number of training rows')
ax.set_xlabel('Number of training rows')
ax.set_ylabel('Accuracy (focused scale)')
ax.set_ylim(0.85, 1.005)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend()
fig.tight_layout()
plt.show()

More training data doesn't guarantee a better score on every step. Here, the large gap between training and validation accuracy is still a warning.

Also look at the dotted baseline. Beating our own previous score is not enough if a model that always says "no" does better!

# 5. Creating a simple fitting curve

Now let's change something DIFFERENT: the tree's complexity, not the amount of data.

We'll use the SAME training and validation rows each time, but change `max_depth`. Remember: a greater depth lets the tree ask more questions on the way to a leaf.

We'll try depths 1 through 25. That is enough to see the pattern without fitting 99 trees! Watch what happens to training accuracy and validation accuracy as the tree gets deeper.

In [ ]:
max_depths = range(1, 26)
depth_train_accuracies = []
depth_validation_accuracies = []

for max_depth in max_depths:
    depth_tree = DecisionTreeClassifier(
        criterion='entropy', max_depth=max_depth, random_state=42
    )
    depth_tree.fit(X_train, Y_train)

    depth_train_accuracies.append(depth_tree.score(X_train, Y_train))
    depth_validation_accuracies.append(depth_tree.score(X_validation, Y_validation))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(max_depths, depth_train_accuracies, 'o-', color='#4C72B0', label='Training')
ax.plot(max_depths, depth_validation_accuracies, 's--', color='#DD8452',
        markerfacecolor='white', label='Validation')
ax.axhline(validation_baseline_accuracy, color='#666666', linestyle=':',
           label='Always no donation (validation)')
ax.set_title('Fitting curve: changing the tree depth')
ax.set_xlabel('Maximum tree depth (model complexity)')
ax.set_ylabel('Accuracy (focused scale)')
ax.set_ylim(0.85, 1.005)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend()
fig.tight_layout()
plt.show()

A deeper tree can fit the training rows more closely. That does NOT mean it predicts new rows better.

For this example, let's choose the depth with the highest VALIDATION accuracy. If several depths tie, we'll take the smallest one. We have NOT looked at the final test score to make this choice.

In [ ]:
# argmax() gives us the position of the first highest score
best_depth_index = int(np.argmax(depth_validation_accuracies))
best_depth = list(max_depths)[best_depth_index]
print('Chosen maximum depth:', best_depth)
print(f'Validation accuracy: {depth_validation_accuracies[best_depth_index]:.1%}')

## 5.1 One final check

We've chosen a depth. Now we can fit a NEW tree with that setting using the training AND validation rows together (`X_development`), then check it on the final test rows.

This is our first look at the final test score. Don't keep changing the model to improve this score; that would turn the test set into another validation set.

In [ ]:
final_tree = DecisionTreeClassifier(
    criterion='entropy', max_depth=best_depth, random_state=42
)
final_tree.fit(X_development, Y_development)
Y_test_predicted = final_tree.predict(X_test)

final_test_accuracy = accuracy_score(Y_test, Y_test_predicted)
final_baseline_accuracy = (Y_test == 0).mean()
final_test_recall = recall_score(Y_test, Y_test_predicted, zero_division=0)

print(f'Final test accuracy: {final_test_accuracy:.1%}')
print(f'Always predict no donation (final test): {final_baseline_accuracy:.1%}')
print(f'People predicted to donate: {int(Y_test_predicted.sum()):,}')
print(f'Recall (donors): {final_test_recall:.1%}')
if Y_test_predicted.sum() > 0:
    print(f'Precision (donors): {precision_score(Y_test, Y_test_predicted):.1%}')
else:
    print('Precision is undefined: the tree predicted no donors.')

Is this a useful donation model? Not just because its accuracy looks high!

If a tree mostly says "no donation," it can look good on accuracy while finding few or no donors. Choosing the best ACCURACY is not the same as choosing the best model for the charity. We'd also need to think about mailing costs, missed donations, and a suitable way to compare models on validation data.

For class, we're using a random split of rows. In a real campaign, we'd also check that the information was available BEFORE the campaign, and that our evaluation reflects the new people or future campaigns we want to predict.

# Bonus: The ROC curve

What if a model gives us a probability, instead of just a yes or no? We can choose how high that probability needs to be before we say "yes."

The **ROC (Receiver Operating Characteristic) curve** shows the true positive rate against the false positive rate as we change that cutoff. More on those names below.

## A short logistic regression introduction

Despite its name, **logistic regression** is used for classification.

In linear regression, we were fitting a weighted sum:

$$z = b_0 + b_1 x_1 + \ldots + b_n x_n$$

Logistic regression puts that weighted sum through a **sigmoid function**:

$$p = \frac{1}{1 + e^{-z}}$$

The result is a number between 0 and 1: the model's estimated probability of class 1. As before, `fit()` learns the coefficients `b_0, b_1, ...` from the training data. These are estimates, not guarantees that the probabilities are perfectly accurate!

Let's look at the sigmoid function first.

In [ ]:
x = np.linspace(-10, 10, 1000)
sigmoid = 1 / (1 + np.exp(-x))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, sigmoid, color='#4C72B0')
ax.set_title('The sigmoid function')
ax.set_xlabel('Input z (the weighted sum)')
ax.set_ylabel('Estimated probability of class 1')
ax.set_ylim(-0.03, 1.03)
fig.tight_layout()
plt.show()

In [ ]:
# change the slope; keep the intercept at 0 in all three examples
fig, ax = plt.subplots(figsize=(8, 4))
for slope, line_style in [(0.5, ':'), (1, '-'), (2, '--')]:
    probability = 1 / (1 + np.exp(-slope * x))
    ax.plot(x, probability, linestyle=line_style, color='#4C72B0',
            label=f'Slope = {slope}')

ax.set_title('Sigmoid functions with different slopes')
ax.set_xlabel('Input x')
ax.set_ylabel('Estimated probability of class 1')
ax.set_ylim(-0.03, 1.03)
ax.legend()
fig.tight_layout()
plt.show()

## Fitting a logistic regression model

Let's create some toy data so we can focus on probabilities and cutoffs. These are NOT donation records, and the two classes are roughly equally common.

We'll use `demo_` names so we don't overwrite our donation data. Again, we split into training, validation, and final test data.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

demo_X, demo_y = make_classification(
    n_samples=1000, n_features=20, n_classes=2, random_state=42
)
demo_X_development, demo_X_test, demo_y_development, demo_y_test = train_test_split(
    demo_X, demo_y, test_size=0.20, stratify=demo_y, random_state=42
)
demo_X_train, demo_X_validation, demo_y_train, demo_y_validation = train_test_split(
    demo_X_development, demo_y_development,
    test_size=0.25, stratify=demo_y_development, random_state=42
)

demo_feature_names = [f'Feature_{i+1}' for i in range(demo_X.shape[1])]
demo_train_df = pd.DataFrame(demo_X_train, columns=demo_feature_names)
demo_train_df['True_Label'] = demo_y_train
demo_validation_df = pd.DataFrame(demo_X_validation, columns=demo_feature_names)
demo_validation_df['True_Label'] = demo_y_validation

demo_train_df.head()

Now let's fit the model to the training data and get probabilities and class predictions.

- `predict_proba()` returns a probability for EACH class. Our columns are ordered `[0, 1]`, so `[:, 1]` selects the probability of class 1.
- `predict()` returns the class itself: 0 or 1. For binary logistic regression, the default cutoff is 0.5.

We'll inspect the VALIDATION data while thinking about cutoffs, not the final test data.

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(demo_X_train, demo_y_train)

print('Probability columns, in order:', clf.classes_)
demo_train_probabilities = clf.predict_proba(demo_X_train)[:, 1]
demo_validation_probabilities = clf.predict_proba(demo_X_validation)[:, 1]
demo_train_predictions = clf.predict(demo_X_train)
demo_validation_predictions = clf.predict(demo_X_validation)

Here are the coefficients we learnt. `coef_` contains one coefficient per feature; `intercept_` contains the constant term.

In [ ]:
print('Intercept:', clf.intercept_[0])
pd.Series(clf.coef_[0], index=demo_feature_names, name='Coefficient')

And here are the predicted labels using the default cutoff of 0.5. Compare the probability, the predicted label, and the actual answer.

In [ ]:
demo_train_df['Predicted_Label'] = demo_train_predictions
demo_train_df['Predicted_Probability'] = demo_train_probabilities
demo_validation_df['Predicted_Label'] = demo_validation_predictions
demo_validation_df['Predicted_Probability'] = demo_validation_probabilities

demo_validation_df[['True_Label', 'Predicted_Label', 'Predicted_Probability']].head(20)

In [ ]:
# choose the most common class using the TRAINING answers, not the validation answers
demo_majority_class = int(pd.Series(demo_y_train).mode().iloc[0])
demo_baseline_predictions = np.full(len(demo_y_validation), demo_majority_class)

print(f'Training accuracy: {accuracy_score(demo_y_train, demo_train_predictions):.1%}')
print(f'Validation accuracy: {accuracy_score(demo_y_validation, demo_validation_predictions):.1%}')
print(f'Always predict class {demo_majority_class} (validation): '
      f'{accuracy_score(demo_y_validation, demo_baseline_predictions):.1%}')

## Choosing a threshold

We don't HAVE to use 0.5. A lower cutoff predicts more positives; a higher cutoff predicts fewer.

Remember our mailing example: a false positive could mean paying to mail someone who won't donate, while a false negative could mean missing a potential donor. Which mistake matters more depends on the costs and benefits.

The ROC curve helps us see the tradeoff:

- **True positive rate** is recall: of the actual positives, what fraction did we find?
- **False positive rate**: of the actual negatives, what fraction did we incorrectly label positive?

We'll use our toy VALIDATION data to look at this. For our own cutoffs below, a probability greater than or equal to the cutoff gets label 1.

In [ ]:
# the full ROC curve, using VALIDATION probabilities
fpr, tpr, roc_thresholds = roc_curve(demo_y_validation, demo_validation_probabilities)
validation_auc = roc_auc_score(demo_y_validation, demo_validation_probabilities)

# compute the EXACT result at each requested cutoff, not the nearest ROC point
selected_thresholds = [1, 0.75, 0.5, 0.25, 0]
threshold_rows = []
for threshold in selected_thresholds:
    threshold_predictions = (demo_validation_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        demo_y_validation, threshold_predictions, labels=[0, 1]
    ).ravel()
    threshold_rows.append({
        'Threshold': threshold,
        'False positives': fp,
        'False negatives': fn,
        'False positive rate': fp / (fp + tn),
        'True positive rate': tp / (tp + fn)
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#4C72B0', label=f'Validation ROC (AUC = {validation_auc:.3f})')
ax.plot([0, 1], [0, 1], color='#666666', linestyle=':', label='Random ranking (in expectation)')
ax.scatter(threshold_results['False positive rate'], threshold_results['True positive rate'],
           color='#4C72B0', edgecolors='white', s=55, zorder=3, label='Exact cutoffs')

# offsets keep the endpoint labels inside the chart and off the curve
label_offsets = [(12, 5), (8, -15), (8, -15), (8, -15), (-38, -16)]
for row, offset in zip(threshold_rows, label_offsets):
    ax.annotate(f"{row['Threshold']:.2f}",
                (row['False positive rate'], row['True positive rate']),
                textcoords='offset points', xytext=offset)

ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate (recall)')
ax.set_title(f'ROC curve on the validation data\nToy data: {len(demo_y_validation)} validation rows')
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

Notice the endpoints: at cutoff 1, this model predicts no positives; at cutoff 0, it predicts everyone positive. The table and the dots use our EXACT cutoffs.

The diagonal is what we'd expect from random ranking. At a given false positive rate, we'd like to find more true positives, so we'd like our curve to be higher.

**AUC** is the area under the ROC curve. It summarizes how well the model ranks positives above negatives. An AUC of 0.9 does NOT mean 90% of its class predictions are correct! It also doesn't tell us which cutoff is best for our costs.

More: [Google's short explanation of ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc).

## One final check for our toy model

Suppose missing a positive is especially costly. Let's pick 0.25 as an ILLUSTRATION of a lower cutoff, then evaluate the fitted model with that fixed cutoff on the final test data.

This is not a claim that 0.25 is the "optimal" cutoff. For a real decision, we'd need the actual costs and benefits, and we'd make that choice using validation data.

We won't refit the toy model here: we want to check the SAME fitted model and cutoff we just considered.

In [ ]:
chosen_threshold = 0.25
demo_test_probabilities = clf.predict_proba(demo_X_test)[:, 1]
demo_test_predictions = (demo_test_probabilities >= chosen_threshold).astype(int)

print('Fixed cutoff:', chosen_threshold)
print(f'Final test accuracy: {accuracy_score(demo_y_test, demo_test_predictions):.1%}')
print(f'Final test precision: {precision_score(demo_y_test, demo_test_predictions, zero_division=0):.1%}')
print(f'Final test recall: {recall_score(demo_y_test, demo_test_predictions, zero_division=0):.1%}')
print(f'Final test AUC: {roc_auc_score(demo_y_test, demo_test_probabilities):.3f}')

pd.DataFrame(
    confusion_matrix(demo_y_test, demo_test_predictions, labels=[0, 1]),
    index=['Actually 0', 'Actually 1'],
    columns=['Predict 0', 'Predict 1']
)

The main idea is the SAME throughout: learn from training data, make choices using validation data, and keep the final test for a fresh check.

And always ask: good compared to WHAT, and good for WHICH decision? :)